In [ ]:
import scipy
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

from sklearn import linear_model    # Herramientas de modelos lineales
from sklearn.metrics import mean_squared_error, r2_score    # Medidas de desempeño
from sklearn.preprocessing import PolynomialFeatures    # Herramientas de polinomios

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.datasets import make_regression
from formulaic import Formula

import matplotlib.pyplot as plt

import formulaic

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Si necesitan instalar algún paquete
#pip install gapminder
#!pip install formulaic

## Colinealidad y explosión de coeficientes

Para cada una de los conjuntos de datos $A$ y $B$, calcular los coeficientes de regresión por mínimos cuadrados al ajustar la variable $y$.

In [ ]:
X_A = np.array([[1], [0.001], [0.001]])
X_B = np.array([[1, 1.001], [0.001, 0.001], [0.001, 0.001]])
y = np.array([1,0,0])
display(X_A)
display(X_B)
display(y)

In [ ]:
# Modelo A
modeloA = linear_model.LinearRegression(fit_intercept = False) 
modeloA.fit(X_A, y)
modeloA.coef_

In [ ]:
# Modelo B
modeloA = linear_model.LinearRegression(fit_intercept = False) 
modeloA.fit(X_B, y)
modeloA.coef_

# Mínimos cuadrados regularizados

Comenzamos primero con un ejemplo sintético que muestra claramente las propiedades y beneficios del método Ridge.

In [ ]:
np.random.seed(42)
n = 150

# Variable latente: "tamaño" de la propiedad
tamanio = np.random.normal(0, 1, n)

# 10 mediciones del mismo tamaño, distintas fuentes/instrumentos
# hay mucha correlación entre las variables (en la simulación son la misma variable con ruido)
ruido = 0.03
X = pd.DataFrame({
    'sup_total_m2':    tamanio + np.random.normal(0, ruido, n),
    'sup_cubierta_m2': tamanio + np.random.normal(0, ruido, n),
    'sup_descubierta': tamanio + np.random.normal(0, ruido, n),
    'sup_escritura':   tamanio + np.random.normal(0, ruido, n),
    'sup_planos':      tamanio + np.random.normal(0, ruido, n),
    'sup_catastro':    tamanio + np.random.normal(0, ruido, n),
    'sup_tasacion':    tamanio + np.random.normal(0, ruido, n),
    'sup_corredor':    tamanio + np.random.normal(0, ruido, n),
    'sup_municipio':   tamanio + np.random.normal(0, ruido, n),
    'sup_satelital':   tamanio + np.random.normal(0, ruido, n),
})

# Precio depende del tamaño real + ruido
y = pd.Series(3 * tamanio + np.random.normal(0, 0.5, n), name='precio_usd')


In [ ]:
X

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

# OLS clásico
ols = LinearRegression()
ols.fit(X_train, y_train)

# Ridge — alpha arbitrario 
ridge = Ridge(alpha=6.5)   # Está también en sklearn.linear_regression
ridge.fit(X_train, y_train)

# Métricas en rain
print("OLS   RMSE train: ", mean_squared_error(y_train, ols.predict(X_train)))
print("Ridge RMSE train: ", mean_squared_error(y_train, ridge.predict(X_train)))
print("---")
# Métricas en TESTEO
print("OLS   RMSE test: ", mean_squared_error(y_test, ols.predict(X_test)))
print("Ridge RMSE test: ", mean_squared_error(y_test, ridge.predict(X_test)))
print("alpha elegido: ", ridge.alpha)

En este ejemplo de juguete todas las variables explicativas son muy parecidas, hay dependencia lineal fuerte.

Analizamos los coeficientes.
    

In [ ]:
# OLS distribuye el peso de forma arbitraria entre variables casi idénticas. 
# Obtenemos coefs grandes y opuestos.
# Ridge los fuerza a ser pequeños y similares entre sí
pd.DataFrame({
    'OLS': ols.coef_,
    'Ridge': ridge.coef_
}, index=X.columns).round(3)

Vemos que en el modelo Ridge todos los coeficientes son positivos y similares, y menores que los coeficientes de OLS.

Podemos pensar que OLS multiplica y agranda los errores, mientras que Ridge los promedia y por lo tanto los cancela entre sí.

**Ejercicio:** Probar distintos valores de alpha, cambian los resultados?

## Boston housing

Analizamos ahora datos reales. 

Consideramos datos de los precios de viviendas en distintos barrios de Boston. Queremos predecir el precio en función de datos demográficos de cada barrio.

In [ ]:
data = pd.read_csv("BostonHousing.csv")
data

| Columna | Descripción |
|---|---|
| `crim` | Tasa de criminalidad per cápita por localidad. |
| `zn` | Proporción de terrenos residenciales zonificados para lotes mayores a 25,000 ft². |
| `indus` | Proporción de acres de negocios no minoristas por localidad. |
| `chas` | Variable binaria: 1 si la localidad limita con el río Charles, 0 en caso contrario. |
| `nox` | Concentración de óxidos nítricos (partes por 10 millones). |
| `rm` | Número promedio de habitaciones por vivienda. |
| `age` | Proporción de viviendas ocupadas por sus propietarios construidas antes de 1940. |
| `dis` | Distancias ponderadas a cinco centros de empleo de Boston. |
| `rad` | Índice de accesibilidad a autopistas radiales. |
| `tax` | Tasa de impuesto a la propiedad por cada \$10,000. |
| `ptratio` | Relación alumno-profesor por localidad. |
| `b` | Variable calculada como \(1000(B_k - 0.63)^2\), donde \(B_k\) es la proporción de población afroamericana. |
| `lstat` | Porcentaje de población de bajo nivel socioeconómico. |
| `medv` | Valor mediano de las viviendas ocupadas por sus propietarios (en miles de dólares). |

## Primero, modelo lineal

In [ ]:
# Consideramos primero un modelo lineal con todas las variables
formula = 'medv ~ crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b  -1'
y, X =  Formula(formula).get_model_matrix(data)

In [ ]:
# Observación: y es una DataFrame (en realidad, una matriz de formulaic)
type(y)

In [ ]:
# Podemos convertirla a serie de Pandas con squeeze() (o iloc[:,0])
# Es conveniente para graficar o acceder a los valores de la serie
y = y.squeeze()
type(y)

In [ ]:
# Separamos en testeo y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Inicializamos el modelo
modeloLineal = linear_model.LinearRegression(fit_intercept = True)  # Como X no hay columna de intercept, lo ponemos acá

# Entrenamos
modeloLineal.fit(X_train, y_train)

# Predecimos
y_pred = modeloLineal.predict(X_test)

# Evaluamos
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error: {rmse:.5f}")

Como tenemos datos poblacionales, puede ser útil considerar interacciones (productos entre variables).
Por ejemplo tiene sentido multiplicar cantidad de habitantes por salario promedio.

Sin pensarlo mucho ni mirar mucho las variables, incorporamos las interacciones entre las variables para ver si podemos mejorar el modelo.

In [ ]:
# En Formulaic agregamos interacciones con el simbolo *.
# De esta forma se agregan las variables individuales y también los productos.
# Más sobre interaccciones en las próximas clases.
formula = 'medv ~ (crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b)*(crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b)-1'
y, X =  Formula(formula).get_model_matrix(data)
y = y.squeeze()
X.head()

In [ ]:
# Separamos en testeo y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Inicializamos el modelo lineal
modeloLineal = linear_model.LinearRegression(fit_intercept = True)

# Entrenamos
modeloLineal.fit(X_train, y_train)

# Predecimos
y_pred = modeloLineal.predict(X_test)

# Evaluamos
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Squared Error: {rmse:.5f}")

Logramos una reducción importante del error cuadrático.

**Ejercicio:** Mirando los coeficientes con cuidado, seleccionar cuáles interacciones son importantes, y utilizar las técnicas vistas de selección de modelos, seleccionar un modelo lineal con pocas variables y similar poder explicativo.

Una forma rápida de mirar coeficientes es graficarlos.

In [ ]:
(
    so.Plot(x = np.arange(X.shape[1]))
    .add(so.Dot(color = "r"), y = modeloLineal.coef_)
)

In [ ]:
modeloLineal.intercept_

Al introducir tantas variables nuevas, relacionadas con las variables originales, es muy posible que hayamos introducido colinealidad entre las variables.

Es razonable entonces intentar un modelo de mínimos cuadrados regularizados.

# Regresión Ridge

**Paso 1:** Separamos los datos en entrenamiento y testeo.

In [ ]:
# Separamos en testeo y entrenamiento
df_train, df_test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# Construimos las matrices X e y para entrenamiento
formula = 'medv ~ (crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b)*(crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b) - 1'
y, X =  Formula(formula).get_model_matrix(df_train)
y = y.squeeze()

**Pregunta:** Que consideran más apropiado:
1. incluir el intecept como columna en los datos (y por lo tanto, incluir el coeficiente $\beta_0$ en la penalización de Ridge)
2. excluir el intecept como columna en los datos (y por lo tanto, excluir el coeficiente $\beta_0$ en la penalización de Ridge)

<details> <summary>Respuesta (click aquí)</summary>
En general debemos excluir el intercept de la penalización. De esta forma nos aseguramos que al tomar valores grandes de $\alpha$, el modelo se aproxime a la recta constante que mejor aproxima los datos.

Si incluimos $\beta_0$ en la penalidad, al tomar $\alpha$ grande, nuestro modelos se a aproxima a la recta $y=0$, que no aproxima los datos.

**Regla rápida:** En modelo Ridge, siempre usar una matriz $X$ sin intercept y la opción `fit_intercept=True`.
</details>

**Paso 2:** Definimos un vector de parámetros a probar

In [ ]:
alphas = np.array([0.001, 0.01, 0.1, 0.5, 1, 2, 3])

**Pasos 3 y 4:** Para cada valor de alpha, calculamos el error promedio al realizar validación cruzada de 5 pliegos en los datos de entrenamiento.

In [ ]:
# Comenzamos con un valor de alpha fijo
alpha = alphas[0]  # alpha = 0.01

cv = KFold(n_splits=5, random_state=42, shuffle=True)  # 5 pliegos

modeloRidge = linear_model.Ridge(alpha = alpha, fit_intercept = ???)    # Inicializamos un modelo de Regresion Lineal con intercept
rmse = np.zeros(cv.get_n_splits())  # Vamos a guardar el error en cada pliego

ind = 0

# Para seleccionar algunas filas dados los índices, utilizamos iloc (lo vimos en la clase 2)
for train_index, val_index in cv.split(X):
    X_train, X_val, y_train, y_val = X.iloc[train_index], X.iloc[val_index], y.iloc[train_index], y.iloc[val_index]
    modeloRidge.fit(???)
    
    y_pred = modeloRidge.predict(???)
    rmse[ind] = np.sqrt(mean_squared_error(y_val, y_pred))
    print("Ridge   RMSE val pliego", ind , ": ", rmse[ind])
    
    ind = ind + 1


print("Ridge RMSE val promedio: ", ???)

Esto lo hicimos para un solor valor de alpha, podemos hacerlo fácilmente para varios valores.

In [ ]:
alphas = np.array([0.001, 0.01, 0.1, 0.5, 1, 2, 3])
cv = KFold(n_splits=5, random_state=2, shuffle=True)  # 5 pliegos

for alpha in alphas:
    # Inicializamos un modelo de Regresion Lineal sin intercept
    modeloRidge = linear_model.Ridge(alpha = alpha, fit_intercept = True)    
    rmse = np.zeros(cv.get_n_splits())  # Vamos a guardar el error en cada pliego

    ind = 0

    # Para seleccionar algunas filas dados los índices, utilizamos iloc (lo vimos en la clase 2)
    for train_index, val_index in cv.split(X):
        X_train, X_val, y_train, y_val = X.iloc[train_index], X.iloc[val_index], y.iloc[train_index], y.iloc[val_index]
        modeloRidge.fit(X_train, y_train)

        y_pred = modeloRidge.predict(X_val)
        rmse[ind] = np.sqrt(mean_squared_error(y_val, y_pred))
        ind = ind + 1

    print(f"Para alfa = {alpha:.5f} el Error Cuadrático Medio promedio es : {rmse.mean():.5f}")   # Algunos trucos para formatear numeros prolijamente

El valor óptimo es alpha = 0.5.
En base a los resultados observados agregamos algunos valores de alpha cercanos a 0.5.

In [ ]:
alphas = np.array([0, 0.001, 0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1, 2, 3])
error_alphas = np.zeros(len(alphas))

cv = KFold(n_splits=5, random_state=2, shuffle=True)  # 5 pliegos

for counter, alpha in enumerate(alphas):  # Truco para tener un contador al recorrer una lista
    modeloRidge = linear_model.Ridge(alpha = alpha, fit_intercept = True)
    rmse = np.zeros(cv.get_n_splits())  # Vamos a guardar el error en cada pliego

    ind = 0

    # Para seleccionar algunas filas dados los índices, utilizamos iloc (lo vimos en la clase 2)
    for train_index, val_index in cv.split(X):
        X_train, X_val, y_train, y_val = X.iloc[train_index], X.iloc[val_index], y.iloc[train_index], y.iloc[val_index]
        modeloRidge.fit(X_train, y_train)

        y_pred = modeloRidge.predict(X_val)
        rmse[ind] = np.sqrt(mean_squared_error(y_val, y_pred))
        ind = ind + 1

    print(f"Para alfa = {alpha:.5f} el Error Cuadrático Medio es : {rmse.mean():.5f}")
    error_alphas[counter] = rmse.mean()

Obtuvimos el valor más chico para $\alpha = 0.4$. 

Fijamos este valor y ajustamos el modelo usando todos los datos.

**Importante:** 
1. Los coeficientes de la regresión son **parámetros** y se recalculan utilizando todos los datos.
2. El coeficiente $\alpha$ es un **hiperparámetro**, queda fijo y no se recalcula.

In [ ]:
alpha_optimo = 0.4

modeloRidge = linear_model.Ridge(alpha = alpha_optimo, fit_intercept = True)    # Inicializamos un modelo de Regresion Lineal sin intercept
modeloRidge.fit(X, y)

Probamos el modelo obtenido en los datos de testeo.

In [ ]:
# Construimos las matrices X e y para testeo
formula = 'medv ~ (crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b)*(crim+zn+indus+chas+nox+rm+age+dis+rad+tax+ptratio+lstat+b)-1'
y_test, X_test =  Formula(formula).get_model_matrix(df_test)
        
y_pred = modeloRidge.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Para alpha = {alpha_optimo:.5f} el RMSE Ridge es : {rmse:.5f}")

# Error OLS
print(f"El RMSE OLS es : {np.sqrt(mean_squared_error(y_test, modeloLineal.predict(X_test))):.5f}")


Obtuvimos un error menor!

Podemos comparar los coeficientes gráficamente:

In [ ]:
(
    so.Plot(x = np.arange(X.shape[1]))
    .add(so.Dot(color = "b"), y = modeloRidge.coef_)
    .add(so.Dot(color = "r"), y = modeloLineal.coef_)
)

In [ ]:
modeloLineal.intercept_

In [ ]:
modeloRidge.intercept_

Observamos que en los coeficientes del modelo lineal hay valores positivos altos que parecen cancelarse con valores negativos altos. Esto suele indicar colinealidad en las variables. Al hacer regresión Ridge, reducimos los problemas de la colinealidad

### Curva de errores 
Puede ser instructivo graficar el error en función de alfa.

In [ ]:
(
    so.Plot(x = ???, y = ???)
    .add(so.Dot(color = "b"))
)

**Preguntas:**
1. Qué pasa si la curva es decreciente?
2. Qué pasa si la curva es creciente?
3. Qué pasa si alpha = 0?

## Ejercicio

En este dataset "datos_simulados.csv" tenemos 400 variables explicativas y 400 observaciones generadas al azar. La variable respuesta $y$ es una combinación lineal con ruido de solo 10 de esas variables.

1. Separar el conjunto de datos en entrenamiento y testeo.
2. Ajustar un modelo de regresión lineal ordinaria en los datos de entrenamiento y medir los errores en testeo. Interpretar la respuesta obtenida.
3. Calcular el hiperparámetro alfa óptimo para regresión Ridge por validación cruzada en los datos de entrenamiento.
4. Para el valor hallado, calcular la fórmula del modelo en los datos de entrenamiento y medir los errores en testeo. ¿Disminuyeron los errores con respecto a mínimos cuadrados ordinarios?
5. Comparar los coeficientes en los métodos obtenidos. 


In [ ]:
df = pd.read_csv("datos_simulados.csv").set_index("n")
df

# Regresión ridge y escalamiento de las variables

Ya vimos que el escalamiento de variables nos puede servir en el modelo lineal para comparar coeficientes del modelo y deducir apropiadamente el peso de cada variable en el modelo.

Sin embargo, el escalamiento no afecta la bondad del modelo. El modelo lineal es invariante por escalamientos lineales de variables.

¿Qué pasa en regresión Ridge? ¿Será invariante por escalamiento de variables?

<details> <summary>Respuesta (click aquí)</summary>
Regresión Ridge NO es invariante por escalamiento. Si algunas variable está en una escala muy alta (por ejemplo medida en pesos) comparada con otras variables (por ejemplo medidas en dólares), el resultado será que las variables en dolares van a ser más penalizadas que las variables en pesos (porque tendrían en general coeficientes mayores).

Por ejemplo, si compro un producto y pago una seña en pesos y el resto en dolares, la formula para calcular el valor total en pesos es
$$total\_pesos = seña\_en\_pesos + 1500 * saldo\_en\_dolares$$
</details>

## Caso de estudio: Jugadores de basketball universitario

In [ ]:
basketball = pd.read_csv("CollegeBasketballPlayers2009-2021.csv")
basketball

In [ ]:
basketball.info()

In [ ]:
# Nos quedamos solo con las variables numéricas
basketNumeric = basketball.select_dtypes(include='number')

### Limpieza de datos: datos faltantes
Veamos cuántos datos faltantes hay por columna

In [ ]:
with pd.option_context('display.max_rows', None): 
    print(basketNumeric.isna().sum())  # Cantidad de datos faltantes por columna

In [ ]:
# Eliminamos primero las columnas con más de 100 datos faltantes
nan_cols = basketNumeric.isna().sum() > 100  # Vector booleano
keep = nan_cols.index[~(nan_cols)] # Lista con los nombres de las columnas para dejar
basketNumeric = basketNumeric[keep] # Seleccionamos solo las columnas en keep

In [ ]:
# Verificamos ahora cuantos datos faltantes hay por columna
basketNumeric.isna().sum()

In [ ]:
# Ahora eliminamos todas las filas con datos faltantes
basketNumericClean = basketNumeric.dropna()
basketNumericClean

Ajutamos primero un modelo lineal sobre todos los datos para predecir la variable pts en función del resto.

In [ ]:
y = basketNumericClean["pts"]

In [ ]:
X = basketNumericClean.drop(["pts"], axis = 1)

In [ ]:
# Inicializamos el modelo lineal
modeloLineal = linear_model.LinearRegression() 

# Entrenamiento
modeloLineal.fit(X, y)

# Predicciones
y_pred = modeloLineal.predict(X)

# Evaluación
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")

In [ ]:
# R cuadrado
r2_score(y, y_pred)

In [ ]:
# Coeficientes
modeloLineal.coef_

In [ ]:
# Graficamos

In [ ]:
so.Plot(x = np.arange(len(modeloLineal.coef_)), y = modeloLineal.coef_).add(so.Dot())

Observando los coeficientes, hay coeficientes grandes que se cancelan, indicando posible colinealidad, lo que sugiere utilizar regresión Ridge. 

Veamos primero si las variables están en la misma escala.

In [ ]:
X.max()

In [ ]:
X.mean()

Vemos que hay mucha diferencia en las escalas de las variables. Para poder comparar mejor los coeficientes, escalamos todas la variables al intervalo [0,1].

Utilizamos el escalamiento estándar. Puede utilizarse también MinMax.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().set_output(transform="pandas") # La última opción hace que nos devuelva un DataFrame

# fit_transform calcula los coeficientes de la transformación y la aplica.
X_scaled = scaler.fit_transform(X)
X_scaled


In [ ]:
X_scaled.max()

In [ ]:
X_scaled.mean()

In [ ]:
# Repetimos el modelo lineal

In [ ]:
# Inicializamos el modelo lineal
modeloLineal = linear_model.LinearRegression() 

# Entrenamiento
modeloLineal.fit(X_scaled, y)

# Predicciones
y_pred = modeloLineal.predict(X_scaled)

# Evaluación
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y, y_pred)
print(f"R cuadrado: {r2:.5f}")

In [ ]:
# Graficamos
so.Plot(x = np.arange(len(modeloLineal.coef_)), y = modeloLineal.coef_).add(so.Dot())

Los problemas de colinealidad son más evidentes ahora.

En base a lo observado, vamos a utilizar un modelo de regresión de Ridge.
Separamos en entrenamiento y testeo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Inicializamos el modelo lineal regularizdo
alpha = 1  # Ejercicio: calcular alpha por validacion cruzada
modeloRidge = linear_model.Ridge(alpha = 1)

In [ ]:
# Entrenamiento
modeloRidge.fit(X_train, y_train)

# Predicciones
y_pred = modeloRidge.predict(X_train)

# Evaluación
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_train, y_pred)
print(f"R cuadrado: {r2:.5f}")

In [ ]:
# Observamos los coeficientes
so.Plot(x = np.arange(len(modeloRidge.coef_)), y = modeloRidge.coef_).add(so.Dot())

Redujimos el problema de coeficientes grandes que se anulan, pero todavía resulta difícil comparar los pesos de las distintas variables. 

Además en el modelo lineal Ridge penalizamos coeficientes grandes. Si las variables están a distinta escala, esto hace que penalicemos más a algunas variables que a otras.

En Regresión Ridge casi siempre es necesario escalar las variables.

Reescalamos todas utilizando StandardScaler

Al realizar un escalamiento, no incluimos los datos de testeo, porque suponemos que son datos desconocidos para nosotros.
StandardScaler nos permite calcular la fórmula de escalamiento en un conjunto de datos y aplicarlo en otro.

In [ ]:
scaler = StandardScaler().set_output(transform="pandas") # La última opción hace que nos devuelva un DataFrame

In [ ]:
# fit_transform calcula los coeficientes de la transformación y la aplica.
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
# Inicializamos el modelo lineal
alpha = 1     # Ejercicio: calcular el alpha optimo por validacion cruzada
modeloRidge = linear_model.Ridge(alpha = alpha)   

# Entrenamiento
modeloRidge.fit(X_train_scaled, y_train)

# Predicciones
y_pred = modeloRidge.predict(X_train_scaled)

# Evaluación
print("alpha = ", alpha)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_train, y_pred)
print(f"R cuadrado: {r2:.5f}")

print(modeloRidge.coef_)

In [ ]:
so.Plot(x = np.arange(len(modeloRidge.coef_)), y = modeloRidge.coef_).add(so.Dot())

Ahora queremos ver los resultados en testeo, para eso transformamos los datos de testeo.

In [ ]:
# Estará bien hacerlo así?
X_test_scaled = scaler.fit_transform(X_test)

In [ ]:
# Predicciones
y_pred = modeloRidge.predict(X_test_scaled)

# Evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_test, y_pred)
print(f"R cuadrado: {r2:.5f}")

No dio muy mal, porque los parámetros del escalamiento son similares (la media y varianza de una muestra es similar a la media varianza de toda la población), pero es incorrecto

La forma correcta es fittear en entrenamiento y aplicar esa transformación a los datos de testeo:

In [ ]:
# Tenemos que fittear en entrenamiento y aplicar esa transformación a los datos de testeo
scaler.fit(???)   # Primero fiteamos (este paso no es necesario si ya hicimos fit_transform en X_train)
X_test_scaled = scaler.transform(???)  # Luego transformamos

In [ ]:
# Predicciones
y_pred = modeloRidge.predict(X_test_scaled)

# Evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_test, y_pred)
print(f"R cuadrado: {r2:.5f}")

Vemos que mejoraron las predicciones. 

La diferencia es más notoria si usamos MinMaxScaler.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().set_output(transform="pandas") # La última opción hace que nos devuelva un DataFrame

In [ ]:
# fit_transform calcula los coeficientes de la transformación y la aplica.
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
# Inicializamos el modelo lineal
alpha = 1
modeloRidge = linear_model.Ridge(alpha = alpha) 

# Entrenamiento
modeloRidge.fit(X_train_scaled, y_train)

In [ ]:
# MAL!

X_test_scaled = scaler.fit_transform(X_test)

# Predicciones
y_pred = modeloRidge.predict(X_test_scaled)

# Evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_test, y_pred)
print(f"R cuadrado: {r2:.5f}")

In [ ]:
# BIEN

# Tenemos que fittear en entrenamiento y aplicar esa transformación a los datos de testeo
scaler.fit(X_train)   # Primero fiteamos (este paso no es necesario si ya hicimos fit_transform en X_train)
X_test_scaled = scaler.transform(X_test)  # Luego transformamos

# Predicciones
y_pred = modeloRidge.predict(X_test_scaled)

# Evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Raíz del Error Cuadrático Medio: {rmse:.5f}")
r2 = r2_score(y_test, y_pred)
print(f"R cuadrado: {r2:.5f}")

## Ejercicio - Repaso
Seleccionar el alpha óptimo por validación cruzada en X_train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)